# NDR — Notebook 7: Reverse-Engineer Trigger + Detection-Aware Repair

**Deliverable:** de-poison WEIGHT thật, đúng tinh thần BTC (co-author-grade). Submit output TRỰC TIẾP.

## Vì sao các NB trước fail (bằng chứng)
NB3-6 đều ĐÈ MÙ (prune/mask/suppress) → firing↔retention tụt cùng nhau. NB6 mask chỉ tắt 1/608 channel.
arXiv 2605.05928 (5/2026, SOTA đúng bài) giải thích gốc:
- 'classification-oriented adversarial generation does not match the detection attack space' → Neural Cleanse/ANP kiểu classifier fail trên detector.
- 'standard detection losses DILUTE the repair signal across many predictions' → fine-tune của ta loãng, không gỡ trúng poison.

## Hướng đúng tiêu chí (co-author)
Baseline CHÍNH THỨC của BTC = Neural Cleanse (reverse-engineer trigger). Ta kết hợp với detection-aware repair (2605.05928):
1. **Reverse-engineer trigger δ:** tối ưu 1 patch nhỏ sao cho dán vào ảnh sạch thì model poisoned BẮN box giả. Loss kiểu BTC Eq.5 (mạnh + bám + không rỗng).
2. **Trigger-aware repair:** dán δ lên ảnh test thật (có pseudo-GT streak), dạy model: vùng δ = background, streak thật GIỮ. Dồn lực sửa vào đúng prediction bị trigger (dual-objective) → KHÔNG đè mù → tách firing khỏi retention.

Đây là de-poison weight thật, tái lập được, đúng baseline BTC.

**Quy tắc nộp:** chỉ nộp khi firing GIẢM THẬT + retention ≥ 0.6 + val_macadd < 1.076 (neo NB4).

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 5.4 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = float(cs[i])-float(ps[j])
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set (poison + synth streak thật)

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference = CHỈ streak thật

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Thước đo genuine + TTA helpers

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


In [8]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7. Baseline poisoned + neo LB

In [9]:
base_stats = eval_model_on_val(poisoned_predictor, VAL)
LB_ANCHOR_MACADD = 1.076   # local val_macadd của p05 (=LB 261.2)
print('POISONED baseline:', {k: round(v,4) for k,v in base_stats.items()})
print('Local neo (neo NB4 = LB258.7): val_macadd =', LB_ANCHOR_MACADD)
print('-> Chỉ nộp NB4 nếu val_macadd < %.3f VÀ retention >= 0.6' % LB_ANCHOR_MACADD)

POISONED baseline: {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
Local neo (neo NB4 = LB258.7): val_macadd = 1.076
-> Chỉ nộp NB4 nếu val_macadd < 1.076 VÀ retention >= 0.6


## 8. Retain-DÀY từ ảnh TEST thật (+synth streak nhãn chuẩn)

In [10]:
IMG_STORE = {}
def build_real_retain(n_test=250, retain_conf=0.4, n_poison_imgs=20, add_synth=True, seed=SEED):
    '''Retain set IN-DISTRIBUTION: nền ảnh test thật + pseudo-GT streak từ model gốc,
    TĂNG mật độ bằng synthetic streak có nhãn chuẩn (chống retain thưa gây under-train).
    Poison patch dán (không nhãn) = forget signal.'''
    rng = np.random.default_rng(seed); records = []; npos = 0; nposimg = 0
    pick = list(range(len(test_files))); rng.shuffle(pick); pick = pick[:n_test]
    for k in pick:
        p = test_files[k]; img = load_image(p); anns = []
        b, sc = predict_image(poisoned_predictor, img)
        rb = b[sc >= retain_conf] if len(b) else b
        for (x1,y1,x2,y2) in rb:
            x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
            x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[x1,y1,x2,y2],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        if add_synth:  # bơm thêm synthetic streak (nhãn CHẮC CHẮN đúng) lên nền test thật
            g = img[:,:,0].copy()
            for _ in range(rng.integers(1,4)):
                g, box = synth_streak_on(g, rng)
                x1,y1,x2,y2 = box
                if x2-x1>1 and y2-y1>1:
                    anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],
                                 'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
            img = np.repeat(g[:,:,None],3,axis=2)
        iid = 20_000_000 + int(k); IMG_STORE[iid] = img.astype(np.float32)
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    # ảnh poison-forget từ unlearn set
    for j in range(n_poison_imgs):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); anns=[]
        b, sc = predict_image(poisoned_predictor, img)
        pb = poison_boxes_by_stem.get(stem, np.zeros((0,4),np.float32))
        keep = sc >= retain_conf
        if len(b) and len(pb) and keep.any():
            keep = keep & (iou_matrix(b, pb).max(1) < POISON_OVERLAP_IOU)
        for (x1,y1,x2,y2) in (b[keep] if len(b) else b):
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        for _ in range(rng.integers(1,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem, ph, pw, rng)
            if spot: paste_patch(img, patch, spot[0], spot[1])
        iid = 21_000_000 + j; IMG_STORE[iid] = img.astype(np.float32); nposimg += 1
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    rng.shuffle(records)
    return records, npos, nposimg
IMG_STORE.clear()
_r,_np,_npi = build_real_retain(n_test=80, n_poison_imgs=10, seed=1)
print(f'probe retain-thật+synth: {len(_r)} ảnh | {_np} pseudo-GT streak ({_np/max(len(_r),1):.2f}/ảnh) | {_npi} ảnh poison-forget')
IMG_STORE.clear()

probe retain-thật+synth: 90 ảnh | 208 pseudo-GT streak (2.31/ảnh) | 10 ảnh poison-forget


## 9. Reverse-engineer trigger δ (Neural Cleanse cho detector, loss kiểu BTC Eq.5)
Tối ưu 1 patch δ (RGB, ~40x40) sao cho dán vào ảnh nền thì model poisoned BẮN detection (điểm cls cao). Forward khả vi qua backbone+head lấy sigmoid cls score. Loss: −α·L_div + λ·‖δ‖ (L_div=chênh score có δ vs không).

In [11]:
def _norm_img(model, x255):
    return (x255 - model.pixel_mean) / model.pixel_std

def _cls_scores(model, x255):
    '''x255: tensor [3,H,W] (RGB,0-255). Trả về vector sigmoid cls score toàn ảnh (khả vi).'''
    nrm = _norm_img(model, x255)[None]
    feats = model.backbone(nrm)
    feats = [feats[f] for f in model.head_in_features]
    logits, _ = model.head(feats)
    scs = [torch.sigmoid(l).flatten() for l in logits]
    return torch.cat(scs)

def reverse_trigger(model, n_bg=12, ph=40, pw=40, iters=300, lr=0.1, alpha=1.0, lam=1e-3, topk=50, seed=SEED):
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    rng = np.random.default_rng(seed)
    # nền = ảnh test có ÍT detection (gần clean) để δ là nguyên nhân bắn box
    idxs = rng.choice(len(test_files), size=min(n_bg*3, len(test_files)), replace=False)
    bgs = []
    for k in idxs:
        img = load_image(test_files[int(k)]); b,s = predict_image(poisoned_predictor, img)
        bgs.append((img, len(s)))
    bgs = [b for b,_ in sorted(bgs, key=lambda t: t[1])[:n_bg]]
    bg_t = [torch.as_tensor(b.transpose(2,0,1), dtype=torch.float32, device=DEVICE) for b in bgs]
    H = W = IMG_H
    delta = torch.zeros(3, ph, pw, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    for it in range(iters):
        opt.zero_grad(); loss = 0.0
        for x in bg_t:
            y0 = int(rng.integers(0, H-ph)); x0 = int(rng.integers(0, W-pw))
            base = _cls_scores(model, x).detach()
            xt = x.clone()
            xt[:, y0:y0+ph, x0:x0+pw] = torch.clamp(delta, 0, 255)
            withd = _cls_scores(model, xt)
            # L_div: top-k score CÓ δ cao hơn KHÔNG δ -> maximize
            ldiv = withd.topk(topk).values.mean() - base.topk(topk).values.mean()
            loss = loss - alpha*ldiv
        loss = loss/len(bg_t) + lam*delta.pow(2).mean()
        loss.backward(); opt.step()
        with torch.no_grad(): delta.clamp_(0, 255)
        if (it+1)%50==0: print(f'  rev-trig it{it+1}: loss={float(loss):.4f}')
    return torch.clamp(delta.detach(),0,255).cpu().numpy().transpose(1,2,0)
print('reverse_trigger defined.')

reverse_trigger defined.


## 10. Kiểm chứng δ: dán vào ảnh sạch có làm model bắn box giả không?
Xác nhận trigger tái tạo đúng (nếu firing tăng rõ khi dán δ -> δ là trigger hợp lệ).

In [12]:
def verify_trigger(delta_np, n=15, seed=1):
    rng = np.random.default_rng(seed); fired_on=fired_off=0; ntot=0
    ph,pw = delta_np.shape[:2]
    idxs = rng.choice(len(test_files), size=n, replace=False)
    for k in idxs:
        img = load_image(test_files[int(k)])
        b0,s0 = predict_image(poisoned_predictor, img)
        y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
        im2 = img.copy(); im2[y0:y0+ph, x0:x0+pw] = delta_np
        b1,s1 = predict_image(poisoned_predictor, im2)
        box = np.array([[x0,y0,x0+pw,y0+ph]],np.float32)
        on  = len(b1) and iou_matrix(box, b1).max()>=0.1 and s1[iou_matrix(box,b1)[0].argmax()]>0.2
        off = len(b0) and iou_matrix(box, b0).max()>=0.1
        fired_on += int(bool(on)); fired_off += int(bool(off)); ntot+=1
    print(f'δ dán vào {ntot} ảnh: bắn box tại δ CÓ trigger={fired_on}/{ntot} | KHÔNG(nền gốc)={fired_off}/{ntot}')
    return fired_on/max(ntot,1)
print('verify_trigger defined.')

verify_trigger defined.


## 11. Dataset retain-thật + dán δ (trigger-augmented) + mapper
Mỗi ảnh retain: xác suất pdrop dán δ vào vị trí ngẫu nhiên NHƯNG không thêm nhãn ở đó -> dạy model 'vùng δ = background'. Streak thật (pseudo-GT) giữ nguyên nhãn -> retention. Đây là trigger-aware repair, không đè mù.

In [13]:
RETAIN_DS='retain_rev'; _REC={'r':None}; TRIG={'d':None,'p':0.5}
def register_dense(records):
    if RETAIN_DS in DatasetCatalog: DatasetCatalog.remove(RETAIN_DS)
    _REC['r']=records; DatasetCatalog.register(RETAIN_DS, lambda: _REC['r'])
    MetadataCatalog.get(RETAIN_DS).set(thing_classes=['object'])
class TrigMapper(DatasetMapper):
    def __call__(self, dd):
        dd=copy.deepcopy(dd); im=IMG_STORE[dd['image_id']].copy()
        if TRIG['d'] is not None and random.random() < TRIG['p']:
            ph,pw = TRIG['d'].shape[:2]
            for _ in range(random.randint(1,3)):
                y0=random.randint(0,IMG_H-ph); x0=random.randint(0,IMG_W-pw)
                im[y0:y0+ph, x0:x0+pw] = TRIG['d']   # dán δ, KHÔNG add nhãn -> forget
        dd['image']=torch.as_tensor(im.transpose(2,0,1).copy())
        dd['instances']=utils.annotations_to_instances(dd.get('annotations',[]), im.shape[:2])
        return dd
print('TrigMapper defined.')

TrigMapper defined.


## 12. Trigger-aware repair — fine-tune nhẹ (backbone freeze, EWC chống trôi)
Loss detection dạy: streak thật giữ (positive), vùng dán δ = background (negative). EWC neo head về model gốc.

In [14]:
from detectron2.utils.events import EventStorage
def freeze_backbone(model):
    for n,p in model.named_parameters():
        if n.startswith('backbone.'): p.requires_grad=False
    return model

def repair(delta_np, n_test, retain_conf, lr, iters, ewc_lambda, drop_p, seed, tag):
    out=WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model=build_model(build_cfg(POISONED_WEIGHTS, output_dir=out))
    DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE); freeze_backbone(model)
    anchor={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
    IMG_STORE.clear(); records,_,_=build_real_retain(n_test=n_test, retain_conf=retain_conf, seed=seed)
    register_dense(records); TRIG['d']=delta_np.astype(np.float32); TRIG['p']=drop_p
    cfg=build_cfg(POISONED_WEIGHTS, output_dir=out)
    cfg.DATASETS.TRAIN=(RETAIN_DS,); cfg.DATALOADER.NUM_WORKERS=2; cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
    loader=build_detection_train_loader(cfg, mapper=TrigMapper(cfg,is_train=True,augmentations=[]),
                                        dataset=DatasetCatalog.get(RETAIN_DS))
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.SGD(params, lr=lr, momentum=0.9)
    model.train(); it=iter(loader)
    with EventStorage(0):
        for step in range(iters):
            try: data=next(it)
            except StopIteration: it=iter(loader); data=next(it)
            ld=model(data); det=sum(v for v in ld.values())
            ewc=torch.tensor(0.0, device=DEVICE)
            for n,p in model.named_parameters():
                if p.requires_grad and n in anchor: ewc=ewc+((p-anchor[n])**2).sum()
            loss=det+ewc_lambda*ewc
            opt.zero_grad(); loss.backward(); opt.step()
    final=str(out/'model_final.pth'); torch.save({'model':model.state_dict()}, final)
    del model; torch.cuda.empty_cache()
    return final
print('repair defined.')

repair defined.


## 13. Chạy reverse-engineer + kiểm chứng trigger

In [15]:
rev_model = build_model(build_cfg(POISONED_WEIGHTS)); DetectionCheckpointer(rev_model).load(POISONED_WEIGHTS); rev_model.to(DEVICE)
DELTA = reverse_trigger(rev_model, n_bg=12, ph=40, pw=40, iters=300, lr=0.1)
del rev_model; torch.cuda.empty_cache()
fire_rate = verify_trigger(DELTA, n=15)
print('-> firing khi dán δ =', round(fire_rate,3), '(cao => δ tái tạo trigger tốt; thấp => cần tăng iters/đổi ph,pw)')

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


  rev-trig it50: loss=-0.0490
  rev-trig it100: loss=-0.0994
  rev-trig it150: loss=-0.1144
  rev-trig it200: loss=-0.1275
  rev-trig it250: loss=-0.1469
  rev-trig it300: loss=-0.1482
δ dán vào 15 ảnh: bắn box tại δ CÓ trigger=1/15 | KHÔNG(nền gốc)=0/15
-> firing khi dán δ = 0.067 (cao => δ tái tạo trigger tốt; thấp => cần tăng iters/đổi ph,pw)


## 14. Sweep repair — chọn theo val_macadd (firing↓ THẬT + retention giữ)

In [16]:
SWEEP=[
    dict(n_test=250, retain_conf=0.4, lr=2e-4, iters=200, ewc_lambda=1000, drop_p=0.5, seed=42, tag='r0'),
    dict(n_test=300, retain_conf=0.4, lr=2e-4, iters=300, ewc_lambda=2000, drop_p=0.6, seed=42, tag='r1'),
    dict(n_test=300, retain_conf=0.4, lr=1e-4, iters=300, ewc_lambda=1000, drop_p=0.7, seed=7,  tag='r2'),
    dict(n_test=250, retain_conf=0.5, lr=2e-4, iters=250, ewc_lambda=3000, drop_p=0.5, seed=13, tag='r3'),
]
sweep_results=[]
for spec in SWEEP:
    w=repair(DELTA, **spec)
    pr=build_predictor(w, CAND_THRESH); stats=eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    sweep_results.append({**spec,'weights':w,**stats})
    print(f"{spec['tag']}: firing={stats['poison_firing_rate']:.3f} retention={stats['streak_retention']:.3f} val_macadd={stats['val_macadd']:.4f}")
sweep_df=pd.DataFrame(sweep_results).sort_values('val_macadd')
print('\n', sweep_df[['tag','n_test','lr','iters','ewc_lambda','drop_p','poison_firing_rate','streak_retention','val_macadd']].to_string(index=False))
print(f"\nPOISONED baseline: firing={base_stats['poison_firing_rate']:.3f} retention={base_stats['streak_retention']:.3f} val_macadd={base_stats['val_macadd']:.4f}")
print(f"neo NB4 (LB258.7): val_macadd={LB_ANCHOR_MACADD}")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


r0: firing=0.969 retention=1.000 val_macadd=1.3988


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


r1: firing=0.939 retention=1.000 val_macadd=1.3668


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


r2: firing=0.969 retention=1.000 val_macadd=1.4170


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


r3: firing=0.969 retention=1.000 val_macadd=1.3941

 tag  n_test     lr  iters  ewc_lambda  drop_p  poison_firing_rate  streak_retention  val_macadd
 r1     300 0.0002    300        2000     0.6            0.938650               1.0    1.366786
 r3     250 0.0002    250        3000     0.5            0.969325               1.0    1.394081
 r0     250 0.0002    200        1000     0.5            0.969325               1.0    1.398757
 r2     300 0.0001    300        1000     0.7            0.969325               1.0    1.417045

POISONED baseline: firing=0.969 retention=1.000 val_macadd=1.4549
neo NB4 (LB258.7): val_macadd=1.076


## 15. Chọn cấu hình + model soup

In [17]:
cands=[r for r in sweep_results if r['streak_retention']>=0.6 and r['val_macadd']<LB_ANCHOR_MACADD]
pool=cands if cands else sweep_results
best=min(pool, key=lambda r:r['val_macadd'])
soup_src=[r for r in sweep_results if r['val_macadd']<LB_ANCHOR_MACADD] or [best]
if len(soup_src)>=2:
    sds=[torch.load(r['weights'],map_location='cpu')['model'] for r in soup_src]
    avg={k:sum(sd[k].float() for sd in sds)/len(sds) for k in sds[0]}
    sw=str(WORK/'soup.pth'); torch.save({'model':avg}, sw)
    pr=build_predictor(sw,CAND_THRESH); ss=eval_model_on_val(pr,VAL); del pr; torch.cuda.empty_cache()
    print('SOUP:', {k:round(v,4) for k,v in ss.items()})
    if ss['streak_retention']>=0.6 and ss['val_macadd']<best['val_macadd']: best={'tag':'soup','weights':sw,**ss}
FINAL_WEIGHTS=best['weights']; FINAL_STATS={k:best[k] for k in ('poison_firing_rate','streak_retention','val_macadd')}
print('CHỌN:', best['tag'], {k:round(float(v),4) for k,v in FINAL_STATS.items()})

CHỌN: r1 {'poison_firing_rate': 0.9387, 'streak_retention': 1.0, 'val_macadd': 1.3668}


## 16. TTA + quét SUBMIT_THRESH (tinh chỉnh cuối, minh bạch)

In [18]:
final_pr=build_predictor(FINAL_WEIGHTS, CAND_THRESH)
tta_stats=eval_model_on_val(final_pr, VAL, tta=True)
USE_TTA = tta_stats['val_macadd'] <= FINAL_STATS['val_macadd']
print('no-TTA:', {k:round(float(v),4) for k,v in FINAL_STATS.items()}, '| TTA:', {k:round(v,4) for k,v in tta_stats.items()}, '-> TTA:', USE_TTA)
def eval_at_thresh(predictor, val, thr):
    pk={}
    for i,item in enumerate(val):
        b,s=predict_image(predictor, item['img']); keep=s>thr
        pk[str(i)]=(b[keep],s[keep]) if len(b) else (b,s)
    return macadd(CLEAN_REF, pk)
grid=[0.2,0.25,0.3,0.35,0.4,0.45,0.5]
tv=[(t,eval_at_thresh(final_pr,VAL,t)) for t in grid]
for t,v in tv: print(f'thr={t}: val_macadd={v:.4f}')
BEST_THRESH=min(tv,key=lambda x:x[1])[0]; print('-> BEST_THRESH=',BEST_THRESH)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


no-TTA: {'poison_firing_rate': 0.9387, 'streak_retention': 1.0, 'val_macadd': 1.3668} | TTA: {'poison_firing_rate': 0.9509, 'streak_retention': 0.9811, 'val_macadd': 2.0614} -> TTA: False
thr=0.2: val_macadd=1.3668
thr=0.25: val_macadd=1.3381
thr=0.3: val_macadd=1.3055
thr=0.35: val_macadd=1.2449
thr=0.4: val_macadd=1.2027
thr=0.45: val_macadd=1.1768
thr=0.5: val_macadd=1.2064
-> BEST_THRESH= 0.45


## 17. Submission = output trực tiếp model đã repair

In [19]:
sample=pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred={}
for p in tqdm(test_files, desc='infer test'):
    img=load_image(p); b,s=predict_tta(final_pr,img) if USE_TTA else predict_image(final_pr,img)
    keep=s>BEST_THRESH; pred[p.stem]=(b[keep],s[keep])
sub=sample.copy(); sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),E)))
sub['prediction_string']=sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string']=' '
sub.to_csv(SUBMISSION_PATH, index=False)
total=sum(len(s) for _,s in pred.values()); npred=int((sub['prediction_string'].str.strip()!='').sum())
print(f'saved {SUBMISSION_PATH} | dets={total} ({total/len(test_files):.3f}/img) @thr={BEST_THRESH} | imgs w/ preds: {npred}')

infer test: 100%|██████████| 2000/2000 [04:07<00:00,  8.09it/s]

saved /kaggle/working/submission.csv | dets=1406 (0.703/img) @thr=0.45 | imgs w/ preds: 1009


## 18. Tổng kết + quy tắc nộp (co-author-grade)

In [20]:
print('='*60)
print('POISONED   :', {k:round(v,4) for k,v in base_stats.items()})
print('DE-POISONED:', {k:round(float(v),4) for k,v in FINAL_STATS.items()})
print('neo NB4    : val_macadd=%.3f (LB=258.7)' % LB_ANCHOR_MACADD)
firing_drop = base_stats['poison_firing_rate'] - FINAL_STATS['poison_firing_rate']
better = FINAL_STATS['val_macadd'] < LB_ANCHOR_MACADD
ret_ok = FINAL_STATS['streak_retention'] >= 0.6
real_depoison = firing_drop >= 0.3   # firing giảm THẬT (không chỉ ngưỡng)
ok = better and ret_ok and real_depoison
print(f'\nfiring giảm thật: {firing_drop:.3f} | retention ok: {ret_ok} | beat neo: {better}')
print('ĐẠT TIÊU CHÍ NỘP (co-author-grade):', ok)
if ok:
    print(' -> NÊN nộp: de-poison WEIGHT thật (firing↓), retention giữ, đúng baseline BTC.')
else:
    print(' -> CHƯA đạt. Nếu δ verify thấp -> tăng iters/đổi ph,pw; nếu firing chưa giảm -> tăng drop_p/iters;')
    print('    nếu retention rớt -> tăng ewc_lambda/n_test.')

POISONED   : {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
DE-POISONED: {'poison_firing_rate': 0.9387, 'streak_retention': 1.0, 'val_macadd': 1.3668}
neo NB4    : val_macadd=1.076 (LB=258.7)

firing giảm thật: 0.031 | retention ok: True | beat neo: False
ĐẠT TIÊU CHÍ NỘP (co-author-grade): False
 -> CHƯA đạt. Nếu δ verify thấp -> tăng iters/đổi ph,pw; nếu firing chưa giảm -> tăng drop_p/iters;
    nếu retention rớt -> tăng ewc_lambda/n_test.
